### Bài thực hành số 4: Phát hiện giao dịch gian lận

Giới thiệu về Dataset:
Bộ dữ liệu IBM Transactions for Anti-Money Laundering (AML) do IBM Research phát triển nhằm giải quyết bài toán thiếu hụt dữ liệu chuẩn (benchmark) trong nghiên cứu phòng chống rửa tiền. Do rào cản bảo mật ngân hàng và quy định quyền riêng tư (GDPR), dữ liệu giao dịch tài chính thật không thể công khai. IBM đã sử dụng các mô phỏng đa tác tử (Multi-Agent Simulation) để tạo ra tập dữ liệu tổng hợp (Synthetic Data) phản ánh chính xác hành vi giao dịch ngân hàng ngoài đời thực.

Tệp HI-Small_trans.csv là phiên bản quy mô nhỏ thuộc dòng tệp HI (High-Involvement - Mức độ phức tạp mạng lưới cao)
+ Quy mô: Khoảng 5 triệu dòng giao dịch (dung lượng chưa nén khoảng 500 MB – 600 MB), vừa đủ gọn để chạy thực hành trên các máy trạm cá nhân hoặc cụm Spark nhỏ.
+ Cấu trúc ma trận mạng: Hàng trăm nghìn tài khoản (Accounts) tạo thành một Đồ thị Hướng Đa cạnh (Directed Multigraph), trong đó 2 tài khoản có thể thực hiện nhiều giao dịch ở các thời điểm khác nhau.

Cấu trúc dữ liệu:
Timestamp: Thời gian thực hiện giao dịch (yyyy/MM/dd HH:mm), dùng để xác định tính tuần tự thời gian giữa các nấc rửa tiền (Temporal Ordering).

From Bank: Mã định danh ngân hàng gửi.

Account2: Mã định danh tài khoản nguồn (src), đóng vai trò nút nguồn trên đồ thị.

To Bank: Mã định danh ngân hàng nhận.

Account4: Mã định danh tài khoản đích (dst), đóng vai trò nút đích trên đồ thị.

Amount Received: Số tiền tài khoản nhận thực tế thu về (sau khi quy đổi hoặc trừ phí).

Receiving Currency: Loại tiền tệ tài khoản nhận thu về.

Amount Paid: Số tiền tài khoản gửi chuyển đi, đặc trưng quan trọng để phát hiện hành vi chia nhỏ số tiền (Structuring).

Payment Currency: Loại tiền tệ dùng khi chuyển đi.

Payment Format: Phương thức thanh toán (ACH, Wire, Cheque, Credit Card, Cash), giúp nhận diện kênh giao dịch có rủi ro cao.

Is Laundering: Nhãn thực tế (Target Label) với 0 là giao dịch hợp lệ và 1 là giao dịch rửa tiền.

+ File dữ liệu thực hành: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml?resource=download&select=HI-Small_Trans.csv

Chuẩn bị môi trường Spark

In [ ]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
from pyspark.sql import functions as F

# Khởi tạo Spark Session tích hợp gói GraphFrames
spark = SparkSession.builder \
    .appName("IBM_AML_Fraud_Detection") \
    .config("spark.local.dir", "D:/spark-temp") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "50")\
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12") \
    .getOrCreate()

Bước 1: Chuẩn bị dữ liệu và Khởi tạo GraphFrame
Đồ thị trong GraphFrames yêu cầu 2 DataFrame chính:
Vertices (Đỉnh): Danh sách các tài khoản ngân hàng, phải có cột định danh duy nhất là id.
Edges (Cạnh): Danh sách các giao dịch, phải có cột nguồn src (tài khoản gửi) và cột đích dst (tài khoản nhận), kèm các thuộc tính như amount, timestamp.

In [ ]:

# Nạp dữ liệu giao dịch IBM (Ví dụ: ibm_transactions.csv)
df_tx = spark.read.csv("HI-Small_Trans.csv", header=True, inferSchema=True)

# 2. Tạo Vertices (Đỉnh): Gom tất cả ID tài khoản từ 'Account2' và 'Account4'
nodes_src = df_tx.select(F.col("Account2").alias("id"))
nodes_dst = df_tx.select(F.col("Account4").alias("id"))
vertices = nodes_src.union(nodes_dst).distinct()

# 3. Tạo Edges (Cạnh): Ánh ánh tên cột về chuẩn của GraphFrame
# Chuyển đổi cột Timestamp từ String sang TimestampType để thực hiện lọc thời gian
edges = df_tx.select(
    F.col("Account2").alias("src"),               # Tài khoản gửi
    F.col("Account4").alias("dst"),               # Tài khoản nhận
    F.col("Amount Paid").alias("amount"),         # Số tiền giao dịch
    F.to_timestamp(F.col("Timestamp"), "yyyy/MM/dd HH:mm").alias("timestamp"), # Ép kiểu thời gian
    F.col("Payment Format").alias("payment_format"),
    F.col("Is Laundering").alias("is_laundering")  # Nhãn gian lận (0 hoặc 1)
)

# 4. Khởi tạo GraphFrame
g = GraphFrame(vertices, edges)

Bước 2: Phát hiện mô hình gian lận bằng Motif Finding
Motif Finding cho phép tìm kiếm các đường đi/mô hình giao dịch bất thường bằng cú pháp khớp chuỗi (DSL).
A. Phát hiện vòng tròn giao dịch (Round-Tripping / Ring Fraud)
Tiền được chuyển lòng vòng qua các tài khoản trung gian để xóa vết (A → B → C → A):

In [3]:
# Tìm chuỗi giao dịch khép kín qua 3 tài khoản khác nhau
cycles = g.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)") \
    .filter("a.id != b.id AND b.id != c.id AND a.id != c.id") \
    .filter("e2.timestamp >= e1.timestamp AND e3.timestamp >= e2.timestamp")

cycles.select("a.id", "b.id", "c.id", "e1.amount", "e2.amount", "e3.amount").show(10)

c:\Users\PC-01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---------+---------+---------+--------+--------+--------+
|       id|       id|       id|  amount|  amount|  amount|
+---------+---------+---------+--------+--------+--------+
|812D22980|811C599A0|811B83280| 5971.98|35687.95|36028.51|
|812D22980|811C599A0|811B83280| 5971.98|35687.95|37764.89|
|812D22980|811C599A0|811B83280| 5971.98|35687.95|40813.17|
|812D22980|811C599A0|811B83280|27247.23|35687.95|36028.51|
|812D22980|811C599A0|811B83280|27247.23|35687.95|37764.89|
|812D22980|811C599A0|811B83280|27247.23|35687.95|40813.17|
|812D22980|811C599A0|811B83280|60056.52|35687.95|36028.51|
|812D22980|811C599A0|811B83280|60056.52|35687.95|37764.89|
|812D22980|811C599A0|811B83280|60056.52|35687.95|40813.17|
|812D22980|811C599A0|811B83280|  655.64|35687.95|36028.51|
+---------+---------+---------+--------+--------+--------+
only showing top 10 rows



B. Phát hiện mô hình gom tiền hoặc phân tán tiền (Scatter-Gather / Smurfing)
Một tài khoản nhận tiền từ nhiều tài khoản khác nhau trong thời gian ngắn, sau đó chuyển toàn bộ sang một tài khoản đích duy nhất:

In [ ]:
# A và B chuyển cho C, sau đó C chuyển ngay cho D
smurfing = g.find("(a)-[e1]->(c); (b)-[e2]->(c); (c)-[e3]->(d)") \
    .filter("a.id != b.id AND c.id != d.id") \
    .filter("e3.timestamp >= e1.timestamp AND e3.timestamp >= e2.timestamp")

smurfing.show(10)

Bài tập 1: Dạng Fan-In (Gom tiền từ nhiều tài khoản nhỏ về 1 tài khoản tập trung)
Mô tả hành vi: Nhiều tài khoản phụ (Smurfs) thực hiện chuyển các khoản tiền nhỏ rải rác về cùng một tài khoản đích duy nhất trong một khoảng thời gian ngắn để gom thành một khoản tiền lớn.

Bài tập 2: Dạng Tài khoản "Lừa" / Tài khoản Trung gian (Pass-through / Mule Account)
Mô tả hành vi: Tài khoản trung gian M (Money Mule) nhận tiền từ tài khoản A, sau đó nhanh chóng chuyển gần như toàn bộ số tiền đó đi cho tài khoản B trong vòng dưới 2 giờ (số tiền giữ lại trong tài khoản $M$ gần như bằng 0).

Bài tập 3: Dạng Chuỗi Phân tầng giảm dần (Layering Chain with Fee Deduction)Mô tả hành vi: Dòng tiền chạy qua chuỗi liên tiếp $A \rightarrow B \rightarrow C \rightarrow D$. Tại mỗi nấc chuyển khoản, số tiền bị khấu trừ một tỷ lệ nhỏ (ví dụ từ $1\%$ đến $5\%$) coi như chi phí hoa hồng cho kẻ trung gian.

Bước 3: Áp dụng thuật toán đồ thị để tính toán chỉ số rủi ro
A. Bậc vào/Bậc ra (Degrees)
Tài khoản có bậc vào/bậc ra cao bất thường thường liên quan đến vai trò tài khoản trung gian (Money Mule) hoặc tài khoản gom tiền.

In [6]:
in_deg = g.inDegrees
out_deg = g.outDegrees

# Tìm các tài khoản nhận số lượng giao dịch áp đảo so với bình thường
high_risk_receivers = in_deg.orderBy("inDegree", ascending=False)

B. PageRank
Xác định độ quan trọng và mức độ tập trung dòng tiền của các tài khoản trong toàn mạng lưới.

In [ ]:
# Chạy PageRank với xác suất ngắt (resetProbability) 0.15
pr_result = g.pageRank(resetProbability=0.15, maxIter=10)
pr_result.vertices.orderBy("pagerank", ascending=False).show(10)

C. Connected Components / Strongly Connected Components
Nhóm các nhóm tài khoản có liên kết chặt chẽ với nhau để phát hiện các băng nhóm/mạng lưới gian lận có tổ chức.

In [ ]:
# Cần cấu hình checkpoint directory cho GraphFrames
spark.sparkContext.setCheckpointDir("/tmp/checkpoints")

components = g.connectedComponents()
# Nhóm theo Component ID để xem quy mô của từng mạng lưới
components.groupBy("component").count().filter("count > 5").show()

Bước 4: Trích xuất Feature cho Machine Learning (MLlib / XGBoost)
Sau khi tính toán các đặc trưng đồ thị, kết hợp chúng lại thành Feature Vector để huấn luyện mô hình phân loại gian lận (Supervised Learning):

In [ ]:
# Tích hợp các chỉ số Đồ thị vào bảng dữ liệu chính
graph_features = vertices.join(g.inDegrees, "id", "left").fillna(0, subset=["inDegree"]) \
                         .join(g.outDegrees, "id", "left").fillna(0, subset=["outDegree"]) \
                         .join(pr_result.vertices.select("id", "pagerank"), "id", "left")

# Đưa graph_features vào các mô hình MLlib (RandomForestClassifier, GBTClassifier) 
# cùng với các thuộc tính giao dịch truyền thống (số tiền, thời gian, thiết bị, v.v.)

Bài tập 4: xây dựng mô hình dự đoán giao dịch có rửa tiền không (dùng thuộc tính Is Laundering giá trị 0 là giao dịch bình thường, 1 là rửa tiền). So sánh độ chính xác với việc huấn luyện mô hình sử dụng các thuộc tính sẵn có, không dùng các đặc trưng của đồ thị